## SETUP 

In [ ]:
## SETUP

import subprocess, sys, os, importlib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pca_main_functions
from sklearn.decomposition import PCA
importlib.reload(pca_main_functions)
from pca_obtain_energy_range import find_common_energy_range
from pca_interpolate_DOS import perform_interpolation


In [ ]:
def overall_pca(results_dir, doscar_dir, dataset):

    energy_ranges = [find_common_energy_range(doscar_dir), (-5.0, 5.0), (-10.0, 10.0)]

    # Initialize an empty list to store results
    results = []

    for range in energy_ranges:
        min_energy_global = range[0]
        max_energy_global = range[1]

        perform_interpolation(doscar_dir, min_energy_global, max_energy_global)

        interpolated_filepath = os.path.join(doscar_dir,'interpolated_dos.txt')

        molecule_names, dos_data = pca_main_functions.load_interpolated_dos(interpolated_filepath)
        
        np.savetxt('molecule_names.txt', molecule_names, fmt='%s')

        energy_values = dos_data.iloc[0,1:]
        dos_samples = dos_data.iloc[1:,:]
        pca, scaled_data = pca_main_functions.perform_PCA(dos_samples)

        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)

        thresholds = [0.80, 0.90]

        for threshold in thresholds:
            n_pc = np.argmax(cumulative_variance >= threshold) + 1
            pca_main_functions.transform_data_to_pc(results_dir, dataset, molecule_names, scaled_data, 
                                                    min_energy_global, max_energy_global, 
                                                    threshold, n_pc)

            # Store the results for this energy range and threshold
            results.append({
                'dataset': dataset,
                'min_energy': min_energy_global,
                'max_energy': max_energy_global,
                'energy_range': f'({min_energy_global:.2f}, {max_energy_global:.2f})',
                'cum_variance': threshold,  # Cumulative variance at the end
                'n_pc': n_pc
            })

    # Convert the results list to a pandas DataFrame
    results_df = pd.DataFrame(results)

    # Save results in overall results excel
    excel_name = os.path.join(results_dir, 'overall_results.xlsx')

    sheet_name = f'pca_results'

    # Save to Excel (append if file already exists)
    if os.path.exists(excel_name):
        # If Excel file does not exists, create a new Excel file
        with pd.ExcelWriter(excel_name, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
            results_df.to_excel(writer, sheet_name=sheet_name, index=False) 
    else:
        with pd.ExcelWriter(excel_name) as writer:
            results_df.to_excel(writer, sheet_name=sheet_name, index=False) 

    return results_df

In [ ]:
# Create an empty DataFrame
overall_results = pd.DataFrame(columns=['dataset',
                                        'energy_range',
                                        'cum_variance',
                                        'n_pc'], dtype='float64')

overall_results

In [ ]:
base_dir = os.path.dirname(os.getcwd())
results_dir =  os.path.join(base_dir, 'RESULTS')

doscar_files_dirs = [f for f in os.listdir(base_dir) if f.startswith('DOSCAR_Files')]

for dir in doscar_files_dirs:
    #print(f"Processing directory is {dir}")
    dir_path = os.path.join(base_dir, dir)
    cur_results = overall_pca(results_dir, dir_path, dir)
    overall_results = pd.concat([overall_results, cur_results], ignore_index=True).drop_duplicates()

overall_results


In [ ]:
# Create a combined 'hue_style' column to combine both cum_variance and energy_range
overall_results['hue_style'] = 'Variance: ' + overall_results['cum_variance'].astype(str) + ' \n Energy range: ' + overall_results['energy_range'].astype(str)

overall_results

In [ ]:
# Create a combined 'hue_style' column to combine both cum_variance and energy_range
overall_results['hue_style'] = 'Variance: ' + overall_results['cum_variance'].astype(str) + ' \n Energy range: ' + overall_results['energy_range'].astype(str)

# Set the style for the plot
sns.set(style="whitegrid")

# Create a scatter plot
plt.figure(figsize=(12, 6))
# sns.scatterplot(data=overall_results, x='dataset', y='n_pc', hue='hue_style', style='hue_style', s=100)
sns.barplot(data=overall_results, x='dataset', y='n_pc', hue='hue_style', dodge=True)

# Add titles and labels
plt.title('Number of Principal Components by Dataset', fontsize=16)
plt.xlabel('Dataset', fontsize=14)
plt.ylabel('Number of Principal Components', fontsize=14)
plt.legend(title='Legend', bbox_to_anchor=(1.05, 1), labelspacing=1.0, loc='upper left')
plt.grid(True)

# Adjust y-axis ticks if necessary
plt.yticks(range(0, int(overall_results['n_pc'].max()) + 5, 5))

# Show the plot
plt.tight_layout()
plt.show()
